In [1]:
import os
from pathlib import Path
import scanpy as sc
from anndata import AnnData as ad

In [4]:
path = "/home/user/Documents/h5adify_bench/h5adify_bench/experiments/data/doi20_modified"
os.listdir(path)

['manifest.json',
 '10.1038_s41586-020-2157-4',
 '10.1038_s41586-020-2496-1',
 '10.1038_s41586-018-0654-5',
 '10.1038_s41586-020-2922-4']

In [ ]:
for folder in os.listdir(path):
    folder_path = os.path.join(path, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        for file in files:
            file_path = os.path.join(folder_path, file)
            if "source" in file_path:
                adata = sc.read_h5ad(file_path)
                adata_min = ad.AnnData(X=adata.X.copy())
                adata_min.obs_names = adata.obs_names
                adata_min.var_names = adata.var_names

                adata_min.write(file_path.replace("source", "empty"))

                manifest["items"].append({
                    "doi": doi,
                    "dataset_id": dataset_id,
                    "dataset_title": title,
                    "collection_name": collection,
                    "organism": organism,
                    "assay": assay,
                    "source_h5ad": str(source_path),
                    "small_h5ad": str(small_path) if small_path else None,
                })

In [ ]:
from scripts.common import ensure_dir, read_yaml, read_json, write_json, doi_slug, now_iso, humanize_exception


path = Path(path)  # convert once
manifest = {
    "created_at": now_iso(),
    "census_version": "stable",
    "items": [],
    "missing": [],
    "errors": [],
}

for file_path in path.rglob("*source*.h5ad"):
    print(file_path)
    adata = sc.read_h5ad(file_path)

    adata_min = ad(X=adata.X.copy())
    adata_min.obs_names = adata.obs_names
    adata_min.var_names = adata.var_names

    new_path = file_path.with_name(file_path.name.replace("source", "empty"))
    adata_min.write(new_path)

    manifest["items"].append({
        "doi": doi,
        "dataset_id": dataset_id,
        "dataset_title": title,
        "collection_name": collection,
        "organism": organism,
        "assay": assay,
        "source_h5ad": str(source_path),
        "small_h5ad": str(small_path) if small_path else None,
    })

/home/user/Documents/h5adify_bench/h5adify_bench/experiments/data/doi20_modified/10.1038_s41586-020-2157-4/2adb1f8a-a6b1-4909-8ee8-484814e2d4bf.source.h5ad
/home/user/Documents/h5adify_bench/h5adify_bench/experiments/data/doi20_modified/10.1038_s41586-020-2496-1/f16a8f4d-bc97-43c5-a2f6-bbda952e4c5c.source.h5ad
/home/user/Documents/h5adify_bench/h5adify_bench/experiments/data/doi20_modified/10.1038_s41586-018-0654-5/28c696bb-9549-434b-9340-dc745a846f9a.source.h5ad
/home/user/Documents/h5adify_bench/h5adify_bench/experiments/data/doi20_modified/10.1038_s41586-020-2922-4/8c42cfd0-0b0a-46d5-910c-fc833d83c45e.source.h5ad


In [ ]:
from scripts.common import ensure_dir, read_yaml, read_json, write_json, doi_slug, now_iso, humanize_exception

manifest_path = os.path.join(path, "manifest.json")
if manifest_path.exists():
    print(f"[i] Resuming from existing manifest: {manifest_path}")
    manifest = read_json(manifest_path)
else:
    manifest = {
        "created_at": now_iso(),
        "census_version": "stable",
        "items": [],
        "missing": [],
        "errors": [],
    }
    manifest.setdefault("in_progress", [])

processed_dois = {
    entry["doi"]
    for section in ("items", "missing", "errors")
    for entry in manifest.get(section, [])
    if "doi" in entry
}

pbar = tqdm(cfg["dois"], desc="DOIs")

for doi in pbar:
    pbar.set_postfix_str(f"DOI: {doi}")
    if doi in processed_dois: continue
    
    manifest["in_progress"].append({
        "doi": doi,
        "started_at": now_iso(),
    })
    write_json(manifest_path, manifest)
    
    try:
        row = find_best_dataset_for_doi(df, doi)
        if row is None:
            manifest["missing"].append({"doi": doi})
            continue

        dataset_id = str(row["dataset_id"])
        title = str(row.get("dataset_title", row.get("title", "")))
        collection = str(row.get("collection_name", ""))
        organism = str(row.get("organism", row.get("organism_name", "")))
        assay = str(row.get("assay", row.get("assay_ontology_term_id", "")))

        doi_dir = out_dir / doi_slug(doi)
        ensure_dir(doi_dir)

        source_path = doi_dir / f"{dataset_id}.source.h5ad"
        download_source_h5ad(dataset_id, source_path, census_version=census_version)

        small_path = None
        if make_small:
            try:
                import anndata as ad
                from common import subsample_adata, subset_vars_by_hvg_or_random
                adata = ad.read_h5ad(source_path)
                adata = subsample_adata(adata, max_obs=small_max_obs, seed=seed)
                adata = subset_vars_by_hvg_or_random(adata, max_vars=small_max_vars, seed=seed)
                small_path = doi_dir / f"{dataset_id}.small.h5ad"
                adata.write_h5ad(small_path)
            except Exception as e:
                manifest["errors"].append({"doi": doi, "dataset_id": dataset_id, "stage": "small_copy", "error": humanize_exception(e)})

        manifest["items"].append({
            "doi": doi,
            "dataset_id": dataset_id,
            "dataset_title": title,
            "collection_name": collection,
            "organism": organism,
            "assay": assay,
            "source_h5ad": str(source_path),
            "small_h5ad": str(small_path) if small_path else None,
        })

    except Exception as e:
        manifest["errors"].append({"doi": doi, "stage": "download", "error": humanize_exception(e)})
        
    finally:
        clear_in_progress(manifest, doi)
        write_json(manifest_path, manifest)

write_json(out_dir / "manifest.json", manifest)
print(f"[ok] Wrote manifest: {out_dir/'manifest.json'}")
print(f"[ok] Downloaded: {len(manifest['items'])}, missing: {len(manifest['missing'])}, errors: {len(manifest['errors'])}")

In [92]:
adata = sc.read_h5ad("../experiments/data/doi20/10.1126_sciadv.adh1914/2/04613344-661b-4b01-8167-00dde7af8065.small.h5ad")

In [93]:
sorted(adata.obs.columns)

['age',
 'assay',
 'assay_ontology_term_id',
 'cell_subcluster',
 'cell_type',
 'cell_type_ontology_term_id',
 'development_stage',
 'development_stage_ontology_term_id',
 'disease',
 'disease_ontology_term_id',
 'donor_id',
 'doublet_score',
 'hemisphere',
 'is_primary_data',
 'isolation_site',
 'n_umi',
 'nuclei_suspension_id',
 'observation_joinid',
 'perc_mitochondrial_umis',
 'region',
 'region_class',
 'region_name',
 'region_subclass',
 'self_reported_ethnicity',
 'self_reported_ethnicity_ontology_term_id',
 'sequencing_run_id',
 'sex',
 'sex_ontology_term_id',
 'social_group',
 'suspension_type',
 'tissue',
 'tissue_ontology_term_id',
 'tissue_type']

In [94]:
adata.uns.keys()

dict_keys(['batch_condition', 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'])

In [95]:
adata.uns['organism']

'Macaca mulatta'

In [91]:
adata.obs["sequencing_run_id"]

NSM354-G04_C05_P04-D03_LIG322    Snyder-Mackler_RNA3-036
NSM354-H02_D01_P04-D05_LIG88     Snyder-Mackler_RNA3-036
NSM354-H02_F08_P04-D05_LIG145    Snyder-Mackler_RNA3-036
NSM354-G03_A06_P04-E01_LIG23     Snyder-Mackler_RNA3-036
NSM328-F02_G12_P01-B03_LIG252    Snyder-Mackler_RNA3-036
                                          ...           
NSM344-E04_H12_P02-F01_LIG260    Snyder-Mackler_RNA3-036
NSM344-G04_E04_P02-F06_LIG106    Snyder-Mackler_RNA3-036
NSM351-H04_F02_P03-F05_LIG11     Snyder-Mackler_RNA3-036
NSM354-E04_E08_P04-E01_LIG314    Snyder-Mackler_RNA3-036
NSM327-F03_D03_P01-A01_LIG299    Snyder-Mackler_RNA3-036
Name: sequencing_run_id, Length: 20000, dtype: category
Categories (3, object): ['Snyder-Mackler_RNA3-019', 'Snyder-Mackler_RNA3-026', 'Snyder-Mackler_RNA3-036']

In [2]:
adata = sc.read_h5ad("/home/user/Documents/h5adify_sex/Soni_Brain_2025_cellxgene_5a6c74b9-da1c-4cef-8fdc-07c7a90089d7.h5ad")

In [3]:
adata

AnnData object with n_obs × n_vars = 94181 × 31671
    obs: 'nCount_RNA', 'nFeature_RNA', 'sub_celltype', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'tissue_type', 'tissue_ontology_term_id', 'sample_id', 'groupid', 'donor_id', 'author_celltype', 'cell_type_ontology_term_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_sampleID_harmony_umap'

In [4]:
adata.obs

,nCount_RNA,nFeature_RNA,sub_celltype,assay_ontology_term_id,development_stage_ontology_term_id,disease_ontology_term_id,is_primary_data,self_reported_ethnicity_ontology_term_id,sex_ontology_term_id,suspension_type,...,author_celltype,cell_type_ontology_term_id,cell_type,assay,disease,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid
index,,,,,,,,,,,,,,,,,,,,,
3920-19_AAACCCACAAACACCT-1,1113.0,591,Astrocyte,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Astrocyte,CL:0000127,astrocyte,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,UrkBeMgX@4
3920-19_AAACCCAGTCCAACGC-1,1417.0,755,Tumor,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Tumor,CL:0001063,neoplastic cell,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,@x{bDyo8WK
3920-19_AAACCCAGTGCGTTTA-1,8532.0,3448,Tumor,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Tumor,CL:0001063,neoplastic cell,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,za~S?77j{3
3920-19_AAACCCAGTGGATACG-1,8987.0,3554,Tumor,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Tumor,CL:0001063,neoplastic cell,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,jm43>$?2c0
3920-19_AAACCCATCCTATTGT-1,1669.0,951,Tumor,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Tumor,CL:0001063,neoplastic cell,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,Y^c3RkVgoa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7484-EGFR_TTTGTTGGTCAAATCC-1,5047.0,2475,Tumor,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Tumor,CL:0001063,neoplastic cell,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,`a@zIa`88l
7484-EGFR_TTTGTTGGTGCTATTG-1,23586.0,5509,Tumor,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Tumor,CL:0001063,neoplastic cell,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,J>VsFaKl5Y
7484-EGFR_TTTGTTGTCAAGATAG-1,19737.0,4337,Tumor,EFO:0009922,MmusDv:0000062,MONDO:0018177,True,na,PATO:0000383,cell,...,Tumor,CL:0001063,neoplastic cell,10x 3' v3,glioblastoma,female,brain,na,2-month-old stage,v^@>4UWM`%


In [8]:
adata.obs['sample_id'].value_counts()

sample_id
5709-TVA-B       15170
5783-TVA-B       14014
5179-WT-HF1      11585
5830-WT-NF1      11021
6791-EGFR        11013
6739-EGFR         9546
7484-EGFR         7280
5310-WT-B-p53     6098
3920-19           5787
5726-TVA-B        2667
Name: count, dtype: int64

In [9]:
adata.obs['donor_id'].value_counts()

donor_id
PDGFB.mGBM.2       15170
PDGFB.mGBM.4       14014
Nf1.mGBM.2         11585
Nf1.mGBM.3         11021
EGFRvIII.mGBM.2    11013
EGFRvIII.mGBM.1     9546
EGFRvIII.mGBM.3     7280
PDGFB.mGBM.1        6098
Nf1.mGBM.1          5787
PDGFB.mGBM.3        2667
Name: count, dtype: int64